In [ ]:
import numpy as np, pandas as pd, os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Experiment C3 — MI Entanglement Metric

The PDF says 'communication is entanglement... mutual information of each agent's world models increases as they develop a shared language.' C3 operationalises this: train two agents with separate world-model encoders and measure the mutual information between their latent states over training using InfoNCE. Hypothesis: MI rises as cross-agent communication improves, even without direct weight sharing.

In [ ]:
!pip install torch matplotlib -q

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
torch.manual_seed(42)

# The PDF (section 8) says:
# "communication is entanglement... mutual information of each agent's world
#  models increases [as they develop a shared language]. The layer of theory
#  of mind between these agents makes it so that socializing is a shared
#  journey through the world."
#
# C3 operationalises this: train two agents with separate WORLD-MODEL encoders.
# Measure the mutual information between their latent states over training.
# Hypothesis: MI rises as a shared language emerges, even without direct
# access to each other's internal states.
# This is the "latent alignment handshake" made quantitative.

N_FEATURES   = 16
N_CANDIDATES = 10
WORLD_DIM    = 32   # world model latent dimension
MSG_DIM      = 16   # message bottleneck
HIDDEN       = 64
N_TRAIN      = 8000
N_VAL        = 2000
N_EPOCHS     = 100
BATCH_SIZE   = 128
MEASURE_EVERY = 5  # measure MI every N epochs
print('Setup complete')

In [ ]:
def make_dataset(n, seed):
    rng = np.random.default_rng(seed)
    objs = rng.integers(0,2,(n,N_CANDIDATES,N_FEATURES)).astype(np.float32)
    return TensorDataset(torch.tensor(objs[:,0,:]), torch.tensor(objs), torch.zeros(n,dtype=torch.long))

train_ds = make_dataset(N_TRAIN, 1)
val_ds   = make_dataset(N_VAL, 2)

# Two agents, each with their own world-model encoder
class AgentEncoder(nn.Module):
    def __init__(self, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.world_model = nn.Sequential(
            nn.Linear(N_FEATURES, HIDDEN), nn.ReLU(),
            nn.Linear(HIDDEN, WORLD_DIM))
        self.msg_head = nn.Sequential(
            nn.Linear(WORLD_DIM, HIDDEN), nn.ReLU(),
            nn.Linear(HIDDEN, MSG_DIM))
    def encode_world(self, x): return self.world_model(x)
    def to_message(self, world_state): return self.msg_head(world_state)

class AgentDecoder(nn.Module):
    def __init__(self, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.msg_proj = nn.Sequential(nn.Linear(MSG_DIM, HIDDEN), nn.ReLU())
        self.obj_proj = nn.Sequential(nn.Linear(N_FEATURES, HIDDEN), nn.ReLU(), nn.Linear(HIDDEN, HIDDEN))
    def forward(self, msg, cands):
        h = self.msg_proj(msg); o = self.obj_proj(cands)
        return torch.bmm(o, h.unsqueeze(-1)).squeeze(-1)

# InfoNCE mutual information estimator
# MI(Z1, Z2) estimated by InfoNCE: lower bound on MI
# High InfoNCE = high MI between the two world models
class InfoNCE_MI(nn.Module):
    def __init__(self, d=WORLD_DIM, temp=0.1):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(d, d*2), nn.ReLU(), nn.Linear(d*2, d))
        self.temp = temp
    def forward(self, z1, z2):
        p1 = F.normalize(self.proj(z1), dim=-1)
        p2 = F.normalize(self.proj(z2), dim=-1)
        B = z1.shape[0]
        logits = (p1 @ p2.T) / self.temp
        labels = torch.arange(B)
        loss = F.cross_entropy(logits, labels)
        # MI lower bound: log(B) - loss
        return np.log(B) - loss.item(), loss

agent1 = AgentEncoder(seed=1); dec1 = AgentDecoder(seed=1)
agent2 = AgentEncoder(seed=2); dec2 = AgentDecoder(seed=2)
mi_est = InfoNCE_MI()

opt = torch.optim.Adam(
    list(agent1.parameters()) + list(agent2.parameters()) +
    list(dec1.parameters()) + list(dec2.parameters()) +
    list(mi_est.parameters()), lr=1e-3)

print('Agents and MI estimator initialised')
print('Agent 1 and Agent 2 have separate world-model encoders (no parameter sharing)')

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
mi_history = []
acc_history = []
epoch_history = []

print('Training with MI measurement every {} epochs...'.format(MEASURE_EVERY))
for epoch in range(N_EPOCHS):
    agent1.train(); agent2.train(); dec1.train(); dec2.train(); mi_est.train()
    ep_accs = []
    for si, cands, labels in DataLoader(train_ds, BATCH_SIZE, shuffle=True):
        # Agent 1: encode world state, send message, agent 2 receives
        w1 = agent1.encode_world(si)
        msg1 = agent1.to_message(w1)
        # Agent 2: same for the same objects (decentralised: no weight sharing)
        w2 = agent2.encode_world(si)
        msg2 = agent2.to_message(w2)

        # Task: each agent uses the OTHER agent's message to identify the target
        scores1 = dec1(msg2, cands)   # agent 1 receives from agent 2
        scores2 = dec2(msg1, cands)   # agent 2 receives from agent 1
        task_loss = F.cross_entropy(scores1, labels) + F.cross_entropy(scores2, labels)

        # MI loss: encourage alignment between world models (optional auxiliary)
        mi_bound, mi_loss = mi_est(w1.detach(), w2.detach())

        # Train on task only (MI is measured, not trained on, to show emergence)
        total_loss = task_loss
        opt.zero_grad(); total_loss.backward(); opt.step()

        acc = ((scores1.argmax(-1)==labels).float().mean() +
               (scores2.argmax(-1)==labels).float().mean()) / 2
        ep_accs.append(acc.item())

    if (epoch+1) % MEASURE_EVERY == 0:
        # Measure MI on validation set
        agent1.eval(); agent2.eval(); mi_est.eval()
        val_mi_vals, val_accs = [], []
        with torch.no_grad():
            for si, cands, labels in DataLoader(val_ds, 256):
                w1 = agent1.encode_world(si)
                w2 = agent2.encode_world(si)
                mi_bound, _ = mi_est(w1, w2)
                val_mi_vals.append(mi_bound)
                msg1 = agent1.to_message(w1)
                msg2 = agent2.to_message(w2)
                s1 = dec1(msg2, cands); s2 = dec2(msg1, cands)
                val_accs.append(((s1.argmax(-1)==labels).float().mean()+
                                  (s2.argmax(-1)==labels).float().mean()).item()/2)
        mi_val = float(np.mean(val_mi_vals))
        acc_val = float(np.mean(val_accs))
        mi_history.append(mi_val)
        acc_history.append(acc_val)
        epoch_history.append(epoch+1)
        print('  Epoch {}: task_acc={:.3f}  MI(w1,w2)={:.3f}'.format(epoch+1, acc_val, mi_val))

# ── Severed baseline: train without cross-agent communication ─────────────────
print('\nTraining severed baseline (no cross-communication)...')
agent1s = AgentEncoder(seed=1); dec1s = AgentDecoder(seed=1)
mi_est_s = InfoNCE_MI()
opt_s = torch.optim.Adam(list(agent1s.parameters())+list(dec1s.parameters())+list(mi_est_s.parameters()), lr=1e-3)
sev_mi_hist, sev_acc_hist = [], []
for epoch in range(N_EPOCHS):
    agent1s.train(); dec1s.train()
    for si, cands, labels in DataLoader(train_ds, BATCH_SIZE, shuffle=True):
        w1 = agent1s.encode_world(si)
        msg1 = agent1s.to_message(w1)
        scores = dec1s(msg1, cands)  # self-communication only
        F.cross_entropy(scores, labels).backward(); opt_s.step(); opt_s.zero_grad()
    if (epoch+1) % MEASURE_EVERY == 0:
        agent1s.eval(); mi_est_s.eval()
        val_mi_s, val_acc_s = [], []
        with torch.no_grad():
            for si, cands, labels in DataLoader(val_ds, 256):
                w1 = agent1s.encode_world(si)
                w2_rand = torch.randn_like(w1)  # random baseline for severed
                mi_b, _ = mi_est_s(w1, w2_rand)
                val_mi_s.append(mi_b)
                s = dec1s(agent1s.to_message(w1), cands)
                val_acc_s.append((s.argmax(-1)==labels).float().mean().item())
        sev_mi_hist.append(float(np.mean(val_mi_s)))
        sev_acc_hist.append(float(np.mean(val_acc_s)))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

ax = axes[0]
ax.plot(epoch_history, mi_history, 'o-', color='#1D9E75', linewidth=2, label='Communicating agents')
ax.plot(epoch_history[:len(sev_mi_hist)], sev_mi_hist, 's--', color='gray', linewidth=2, label='Severed (random MI)')
ax.set_xlabel('Training epoch'); ax.set_ylabel('MI lower bound (InfoNCE)')
ax.set_title('MI between agent world models over training\n(does communication create entanglement?)')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(epoch_history, acc_history, 'o-', color='#1D9E75', linewidth=2, label='Cross-agent accuracy')
ax.plot(epoch_history[:len(sev_acc_hist)], sev_acc_hist, 's--', color='gray', linewidth=2, label='Self-comm baseline')
ax.set_xlabel('Training epoch'); ax.set_ylabel('Task accuracy')
ax.set_title('Task accuracy over training\n(validates communication is happening)')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
ax.scatter(acc_history, mi_history, s=60, c=epoch_history, cmap='viridis', zorder=3)
for i, ep in enumerate(epoch_history):
    if ep % 25 == 0:
        ax.annotate('ep{}'.format(ep), (acc_history[i], mi_history[i]), fontsize=8)
plt.colorbar(ax.collections[0], ax=ax, label='Epoch')
ax.set_xlabel('Task accuracy'); ax.set_ylabel('MI lower bound')
ax.set_title('MI vs accuracy trajectory\n(do they co-emerge?)')
ax.grid(True, alpha=0.3)

plt.suptitle('Experiment C3 — MI Entanglement Metric\n'
             'Does mutual information between agent world models rise as shared language emerges?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp_c3_mi_entanglement.png', dpi=150, bbox_inches='tight')
plt.show()

mi_rise = mi_history[-1] - mi_history[0] if mi_history else 0
print('='*60)
print('EXPERIMENT C3 — SUMMARY')
print('='*60)
print('MI rise over training: {:.3f} (communicating agents)'.format(mi_rise))
if len(sev_mi_hist) >= 2:
    sev_rise = sev_mi_hist[-1] - sev_mi_hist[0]
    print('MI rise (severed baseline): {:.3f}'.format(sev_rise))
    if mi_rise > sev_rise + 0.1:
        print('VERDICT: MI RISES with communication — entanglement is real.')
        print('The shared language creation increases world-model alignment.')
    else:
        print('VERDICT: MI rise is comparable to baseline — entanglement not clearly detected.')
        print('Try longer training or a more expressive world model encoder.')
print()
print('PDF prediction: "communication is entanglement... MI of world models increases"')